In [1]:
# [COLAB SETUP]
import sys
import os

if "google.colab" in sys.modules:
    print("Running in Google Colab. Setting up environment...")
    
    # Mount Google Drive to persist the datasets and cloned repository
    from google.colab import drive
    drive.mount('/content/drive')
    
    repo_path = '/content/drive/MyDrive/Sinhala-Script-Language-Identification-LangID-for-Sinhala-Pali-and-Sanskrit'
    
    if not os.path.exists(repo_path):
        print(f"Cloning repository into {repo_path}...")
        os.makedirs('/content/drive/MyDrive', exist_ok=True)
        os.system(f'git clone https://github.com/Maleesha-K/Sinhala-Script-Language-Identification-LangID-for-Sinhala-Pali-and-Sanskrit.git {repo_path}')
        
    os.chdir(repo_path + '/data_pipeline')
    print("Installing dependencies...")
    os.system('pip install -q pandas scikit-learn fasttext huggingface_hub gdown')
    print("Setup complete!")


In [2]:
input_file = 'datasets/finetuning/train.csv'
benchmark_dir = 'datasets/preprocessed'
output_dir = 'models/finetuned/GlotLID_v3'


In [3]:
import os
# Auto-resolve the project root if running manually
if not os.path.exists("Makefile") and os.path.exists("../../Makefile"):
    os.chdir("../../")

import json
import glob
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, f1_score
import fasttext
from huggingface_hub import hf_hub_download

MODEL_NAME = "GlotLID v3"
MODEL_ID = "glotlid_v3"


In [4]:
print("Downloading GlotLID v3 base model from Hugging Face (~1.6GB)...")
base_model_path = hf_hub_download(repo_id="cis-lmu/glotlid", filename="model.bin")
base_model_dir = os.path.dirname(base_model_path)

print(f"Loading base GlotLID v3 model from {base_model_path}...")
base_model = fasttext.load_model(base_model_path)

# Extract pretrained word vectors for transfer learning / fine-tuning
vec_file_path = os.path.join(base_model_dir, "glotlid_v3.vec")
if not os.path.exists(vec_file_path):
    print("Extracting pre-trained word vectors to .vec format...")
    words = base_model.get_words()
    dim = base_model.get_dimension()
    with open(vec_file_path, "w", encoding="utf-8") as f:
        f.write(f"{len(words)} {dim}\n")
        for word in words:
            v_str = " ".join(map(str, base_model.get_word_vector(word)))
            f.write(f"{word} {v_str}\n")
    print(f"Extracted {len(words)} vectors into {vec_file_path}")


Loading base GlotLID v3 model from C:\Users\USER\.cache\huggingface\hub\models--cis-lmu--glotlid\snapshots\85cd6716494360367b75f642b5bc78667605d0b4\model.bin...


In [5]:
print(f"Loading finetuning dataset from {input_file}...")
df = pd.read_csv(input_file)
print(f"Loaded {len(df)} rows.")

def format_text(text):
    return str(text).replace("\n", " ").strip()

df["clean_text"] = df["text"].apply(format_text)
df["ft_line"] = "__label__" + df["label"].astype(str) + " " + df["clean_text"]

train_df, val_df = train_test_split(df, test_size=0.1, random_state=42, stratify=df["label"])

os.makedirs(output_dir, exist_ok=True)
train_ft_file = os.path.join(output_dir, "train_formatted.txt")
val_ft_file = os.path.join(output_dir, "val_formatted.txt")

with open(train_ft_file, "w", encoding="utf-8") as f:
    f.write("\n".join(train_df["ft_line"].tolist()) + "\n")

with open(val_ft_file, "w", encoding="utf-8") as f:
    f.write("\n".join(val_df["ft_line"].tolist()) + "\n")

print(f"Saved {len(train_df)} training samples to {train_ft_file}")
print(f"Saved {len(val_df)} validation samples to {val_ft_file}")


Loading finetuning dataset from datasets/finetuning/train.csv...
Loaded 60285 rows.
Saved 54256 training samples to models/finetuned/GlotLID_v3\train_formatted.txt
Saved 6029 validation samples to models/finetuned/GlotLID_v3\val_formatted.txt


In [6]:
print(f"Starting fine-tuning for {MODEL_NAME}...")
finetuned_model = fasttext.train_supervised(
    input=train_ft_file,
    pretrainedVectors=vec_file_path,
    dim=base_model.get_dimension(),
    epoch=25,
    lr=0.5,
    wordNgrams=2,
    loss="softmax"
)

save_model_path = os.path.join(output_dir, f"{MODEL_ID}_finetuned.bin")
finetuned_model.save_model(save_model_path)
print(f"Fine-tuned model successfully saved to {save_model_path}")


Starting fine-tuning for GlotLID v3...
Fine-tuned model successfully saved to models/finetuned/GlotLID_v3\glotlid_v3_finetuned.bin


In [7]:
val_texts = val_df["clean_text"].tolist()
val_true = val_df["label"].astype(str).tolist()

print(f"Evaluating fine-tuned {MODEL_NAME} on {len(val_texts)} validation samples...")
preds, _ = finetuned_model.predict(val_texts, k=1)
val_pred = [p[0].replace("__label__", "") for p in preds]

acc = accuracy_score(val_true, val_pred)
macro_f1 = f1_score(val_true, val_pred, average="macro")

print("\n" + "=" * 48)
print(f"VALIDATION FINE-TUNING RESULTS ({MODEL_NAME})")
print("=" * 48)
print(f"Accuracy:  {acc * 100:.2f}%")
print(f"Macro F1:  {macro_f1 * 100:.2f}%")
print("=" * 48)
print("\nPer-language breakdown:\n")
print(classification_report(val_true, val_pred, digits=4, zero_division=0))


Evaluating fine-tuned GlotLID v3 on 6029 validation samples...

VALIDATION FINE-TUNING RESULTS (GlotLID v3)
Accuracy:  99.00%
Macro F1:  98.93%

Per-language breakdown:

              precision    recall  f1-score   support

        pali     0.9819    0.9936    0.9877      2349
    sanskrit     0.9940    0.9802    0.9871      1012
     sinhala     0.9959    0.9906    0.9932      2668

    accuracy                         0.9900      6029
   macro avg     0.9906    0.9882    0.9893      6029
weighted avg     0.9901    0.9900    0.9901      6029



In [8]:
print(f"Evaluating fine-tuned {MODEL_NAME} on Sinhala script target languages across benchmark datasets in {benchmark_dir}...")

TARGET_LANGUAGES = ["sinhala", "pali", "sanskrit"]

def map_benchmark_label(row):
    lbl = row.get("label")
    src = row.get("source")
    if lbl in ["sin", "sin_Sinh", "sinhala", "si"]:
        return "sinhala"
    if lbl in ["pli", "pli_Sinh", "pali", "pi"]:
        return "pali"
    if lbl in ["san_Sinh", "sanskrit"] or (lbl == "san" and src in ["DCS", "SansinNT", "SiDiaC-v2"]):
        return "sanskrit"
    return None

def load_benchmark_dataset(file_path):
    records = []
    with open(file_path, encoding="utf-8") as f:
        for line in f:
            row = json.loads(line)
            mapped_label = map_benchmark_label(row)
            if mapped_label:
                row["target_label"] = mapped_label
                records.append(row)
    return pd.DataFrame(records)

benchmark_files = sorted(glob.glob(os.path.join(benchmark_dir, "*.jsonl")))
if not benchmark_files:
    print(f"No benchmark datasets found in {benchmark_dir}.")
else:
    results_dir = os.path.join("datasets", "benchmark_results")
    os.makedirs(results_dir, exist_ok=True)
    
    for file_path in benchmark_files:
        dataset_name = os.path.splitext(os.path.basename(file_path))[0]
        df_bench = load_benchmark_dataset(file_path)
        if df_bench.empty:
            print(f"No matching target languages found in {dataset_name}.")
            continue
        
        texts = df_bench["text"].apply(format_text).tolist()
        print(f"\nEvaluating {len(texts)} target language samples from {dataset_name}...")
        preds, _ = finetuned_model.predict(texts, k=1)
        
        results = df_bench[["text", "label", "source"]].copy()
        results["true_label"] = df_bench["target_label"]
        results["predicted_label"] = [p[0].replace("__label__", "") for p in preds]
        
        acc_b = accuracy_score(results["true_label"], results["predicted_label"])
        macro_f1_b = f1_score(
            results["true_label"], results["predicted_label"],
            average="macro", labels=TARGET_LANGUAGES, zero_division=0
        )
        
        print("=" * 65)
        print(f"BENCHMARK RESULTS ({MODEL_NAME} Finetuned on train.csv - Evaluated on {dataset_name})")
        print("=" * 65)
        print(f"Accuracy:  {acc_b * 100:.2f}%")
        print(f"Macro F1:  {macro_f1_b * 100:.2f}%")
        print("=" * 65)
        print("\nPer-language breakdown (F1 scores & metrics for target languages):\n")
        print(classification_report(
            results["true_label"], results["predicted_label"],
            labels=TARGET_LANGUAGES, digits=4, zero_division=0
        ))
        
        out_csv = os.path.join(results_dir, f"{MODEL_ID}_finetuned_{dataset_name}.csv")
        results.to_csv(out_csv, index=False)
        print(f"Saved benchmark predictions to {out_csv}")


Evaluating fine-tuned GlotLID v3 on Sinhala script target languages across benchmark datasets in datasets/preprocessed...

Evaluating 7047 target language samples from commonlid...
BENCHMARK RESULTS (GlotLID v3 Finetuned on train.csv - Evaluated on commonlid)
Accuracy:  98.96%
Macro F1:  98.84%

Per-language breakdown (F1 scores & metrics for target languages):

              precision    recall  f1-score   support

     sinhala     0.9915    0.9955    0.9935      2693
        pali     0.9853    0.9934    0.9893      3027
    sanskrit     0.9961    0.9691    0.9824      1327

    accuracy                         0.9896      7047
   macro avg     0.9910    0.9860    0.9884      7047
weighted avg     0.9897    0.9896    0.9896      7047

Saved benchmark predictions to datasets\benchmark_results\glotlid_v3_finetuned_commonlid.csv

Evaluating 7047 target language samples from flores_plus...
BENCHMARK RESULTS (GlotLID v3 Finetuned on train.csv - Evaluated on flores_plus)
Accuracy:  98.96%
M

In [9]:
print(f"Evaluating fine-tuned {MODEL_NAME} across ALL benchmark languages in {benchmark_dir}...")

ALL_BENCHMARK_LANGUAGES = [
    "sinhala", "pali", "sanskrit", "sanskrit_deva", "english", "tamil",
    "hindi", "bengali", "arabic", "french", "german"
]

LABEL_MAPPING_ALL = {
    "sin": "sinhala", "sin_Sinh": "sinhala", "sinhala": "sinhala", "si": "sinhala",
    "pli": "pali", "pli_Sinh": "pali", "pli_Latn": "pali", "pali": "pali", "pi": "pali",
    "san_Sinh": "sanskrit",
    "san_Deva": "sanskrit_deva", "sa": "sanskrit_deva",
    "eng": "english", "eng_Latn": "english", "english": "english", "en": "english",
    "tam": "tamil", "tam_Taml": "tamil", "tamil": "tamil", "ta": "tamil",
    "hin": "hindi", "hin_Deva": "hindi", "hindi": "hindi", "hi": "hindi",
    "ben": "bengali", "ben_Beng": "bengali", "bengali": "bengali", "bn": "bengali",
    "arb": "arabic", "arb_Arab": "arabic", "arabic": "arabic", "ar": "arabic",
    "fra": "french", "fra_Latn": "french", "french": "french", "fr": "french",
    "deu": "german", "deu_Latn": "german", "german": "german", "de": "german"
}

def map_all_label(row):
    lbl = row.get("label")
    src = row.get("source")
    if lbl == "san":
        if src in ["DCS", "SansinNT", "SiDiaC-v2"]:
            return "sanskrit"
        else:
            return "sanskrit_deva"
    return LABEL_MAPPING_ALL.get(lbl)

def load_all_languages_dataset(file_path):
    records = []
    with open(file_path, encoding="utf-8") as f:
        for line in f:
            row = json.loads(line)
            mapped_label = map_all_label(row)
            if mapped_label:
                row["target_label"] = mapped_label
                records.append(row)
    return pd.DataFrame(records)

benchmark_files = sorted(glob.glob(os.path.join(benchmark_dir, "*.jsonl")))
if not benchmark_files:
    print(f"No benchmark datasets found in {benchmark_dir}.")
else:
    results_dir = os.path.join("datasets", "benchmark_results")
    os.makedirs(results_dir, exist_ok=True)
    
    for file_path in benchmark_files:
        dataset_name = os.path.splitext(os.path.basename(file_path))[0]
        df_all = load_all_languages_dataset(file_path)
        if df_all.empty:
            print(f"No matching languages found in {dataset_name}.")
            continue
        
        texts = df_all["text"].apply(format_text).tolist()
        print(f"\nEvaluating {len(texts)} samples across ALL benchmark languages from {dataset_name}...")
        preds, _ = finetuned_model.predict(texts, k=1)
        
        results = df_all[["text", "label", "source"]].copy()
        results["true_label"] = df_all["target_label"]
        results["predicted_label"] = [p[0].replace("__label__", "") for p in preds]
        
        acc_all = accuracy_score(results["true_label"], results["predicted_label"])
        macro_f1_all = f1_score(
            results["true_label"], results["predicted_label"],
            average="macro", labels=ALL_BENCHMARK_LANGUAGES, zero_division=0
        )
        
        print("=" * 65)
        print(f"ALL LANGUAGES BENCHMARK RESULTS ({MODEL_NAME} Finetuned on train.csv - Evaluated on {dataset_name})")
        print("=" * 65)
        print(f"Accuracy:  {acc_all * 100:.2f}%")
        print(f"Macro F1:  {macro_f1_all * 100:.2f}%")
        print("=" * 65)
        print("\nPer-language breakdown (All Benchmark Languages):\n")
        print(classification_report(
            results["true_label"], results["predicted_label"],
            labels=ALL_BENCHMARK_LANGUAGES, digits=4, zero_division=0
        ))
        
        out_csv = os.path.join(results_dir, f"{MODEL_ID}_finetuned_all_langs_{dataset_name}.csv")
        results.to_csv(out_csv, index=False)
        print(f"Saved all-languages benchmark predictions to {out_csv}")


Evaluating fine-tuned GlotLID v3 across ALL benchmark languages in datasets/preprocessed...

Evaluating 77974 samples across ALL benchmark languages from commonlid...
ALL LANGUAGES BENCHMARK RESULTS (GlotLID v3 Finetuned on train.csv - Evaluated on commonlid)
Accuracy:  8.94%
Macro F1:  6.64%

Per-language breakdown (All Benchmark Languages):

               precision    recall  f1-score   support

      sinhala     0.0716    0.9955    0.1336      2693
         pali     0.0834    0.9934    0.1538      3027
     sanskrit     0.2872    0.9691    0.4431      1327
sanskrit_deva     0.0000    0.0000    0.0000       895
      english     0.0000    0.0000    0.0000     27461
        tamil     0.0000    0.0000    0.0000        81
        hindi     0.0000    0.0000    0.0000      3666
      bengali     0.0000    0.0000    0.0000      1886
       arabic     0.0000    0.0000    0.0000     26152
       french     0.0000    0.0000    0.0000      3233
       german     0.0000    0.0000    0.0000    

In [10]:
print("\n" + "=" * 50)
print(f"FINE-TUNING & BENCHMARKING COMPLETE FOR {MODEL_NAME}")
print(f"Fine-tuned model saved to: {save_model_path}")
print("=" * 50)



FINE-TUNING & BENCHMARKING COMPLETE FOR GlotLID v3
Fine-tuned model saved to: models/finetuned/GlotLID_v3\glotlid_v3_finetuned.bin


### Candidate model verification

The fine-tuning cells above train a **3-label** model (`glotlid_v3_finetuned.bin`) which
cannot score the non-target languages — hence the near-zero all-language results.

The reported results table was produced by one of the **2104-label** checkpoints in
`models/finetuned/GlotLID_v3/` (GlotLID's 2102 labels + `pli_Sinh` + `san_Sinh`).
The cells below evaluate every candidate `.bin` across all three benchmarks and
identify which one reproduces the reported per-language F1 scores.


In [ ]:
# ============================================================
# VERIFY: evaluate every candidate .bin across all benchmarks
# ============================================================
import os, json, glob, gc
import fasttext
import pandas as pd
from sklearn.metrics import accuracy_score, f1_score

CANDIDATE_DIR = "models/finetuned/GlotLID_v3"
BENCH_DIR = benchmark_dir

# GlotLID emits ISO 639-3 + script codes; map them to evaluation names.
PRED_MAPPING = {
    "sin_Sinh": "sinhala",
    "pli_Sinh": "pali",
    "san_Sinh": "sanskrit",
    "san_Deva": "sanskrit_deva",
    "eng_Latn": "english",
    "tam_Taml": "tamil",
    "hin_Deva": "hindi",
    "ben_Beng": "bengali",
    "arb_Arab": "arabic",
    "fra_Latn": "french",
    "deu_Latn": "german",
}

# Reuse map_all_label / load_all_languages_dataset / ALL_BENCHMARK_LANGUAGES
# from the all-languages benchmark cell above.
benchmarks = {}
for fp in sorted(glob.glob(os.path.join(BENCH_DIR, "*.jsonl"))):
    name = os.path.splitext(os.path.basename(fp))[0]
    benchmarks[name] = load_all_languages_dataset(fp)
    print(f"Loaded {name}: {len(benchmarks[name])} samples")

candidate_results = {}

for model_path in sorted(glob.glob(os.path.join(CANDIDATE_DIR, "*.bin"))):
    model_name = os.path.basename(model_path)
    print("\n" + "=" * 78)
    print(f"MODEL: {model_name}")
    print("=" * 78)

    candidate = fasttext.load_model(model_path)
    n_labels = len(candidate.get_labels())

    if n_labels < 100:
        print(f"  SKIP - only {n_labels} labels (3-class model, cannot score all languages)")
        del candidate
        gc.collect()
        continue

    print(f"  labels={n_labels}  dim={candidate.get_dimension()}")

    for bench_name, df_b in benchmarks.items():
        texts = df_b["text"].apply(format_text).tolist()
        preds, _ = candidate.predict(texts, k=1)
        raw = [p[0].replace("__label__", "") for p in preds]

        y_pred = [PRED_MAPPING.get(r, "other") for r in raw]
        y_true = df_b["target_label"].tolist()

        f1s = f1_score(y_true, y_pred, labels=ALL_BENCHMARK_LANGUAGES,
                       average=None, zero_division=0)
        macro = f1_score(y_true, y_pred, labels=ALL_BENCHMARK_LANGUAGES,
                         average="macro", zero_division=0)
        acc = accuracy_score(y_true, y_pred)

        candidate_results[(model_name, bench_name)] = (list(f1s), macro, acc)

        print(f"\n  -- {bench_name}  (n={len(df_b)})  acc={acc:.4f}  macroF1={macro:.4f}")
        supports = df_b["target_label"].value_counts().to_dict()
        for lang, f in zip(ALL_BENCHMARK_LANGUAGES, f1s):
            print(f"       {lang:15s} {f:.4f}  (support={supports.get(lang, 0)})")

    del candidate
    gc.collect()

print("\nEvaluated", len(candidate_results), "model/benchmark combinations.")


In [ ]:
# ============================================================
# VERIFY: match candidate scores against the reported table
# ============================================================

# Per-language F1 rows as they appear in the reported results table,
# in ALL_BENCHMARK_LANGUAGES order. None = blank cell (no support).
REPORTED_ROWS = {
    "row1": [0.9688, 0.9566, 0.9019, 0.9955, 1.0, 1.0, 0.9926, 0.9995, 0.5962, 1.0, 1.0],
    "row2": [0.9589, 0.9566, 0.9019, 0.9612, 0.9270, 0.9818, 0.9646, 0.9792, 0.9603, 0.9256, 0.9211],
    "row3": [0.9688, 0.9566, 0.9019, 0.9914, 0.9367, 0.9950, 0.9126, 0.9451, None, 0.9875, 0.9744],
}

TOLERANCE = 0.0002

print("=" * 78)
print("MATCHING CANDIDATES AGAINST THE REPORTED TABLE")
print("=" * 78)

winners = {}

for row_name, target in REPORTED_ROWS.items():
    print(f"\n=== {row_name} ===")
    best = None

    for (model_name, bench_name), (f1s, macro, acc) in candidate_results.items():
        diffs = [abs(t - g) for t, g in zip(target, f1s) if t is not None]
        if not diffs:
            continue
        max_diff = max(diffs)
        mean_diff = sum(diffs) / len(diffs)

        if best is None or max_diff < best[0]:
            best = (max_diff, model_name, bench_name, macro)

        flag = "  <== MATCH" if max_diff < TOLERANCE else ""
        print(f"  {model_name:38s} {bench_name:13s} "
              f"maxdiff={max_diff:.4f} mean={mean_diff:.4f} macro={macro:.4f}{flag}")

    if best:
        winners[row_name] = best
        print(f"  BEST: {best[1]} / {best[2]}  maxdiff={best[0]:.4f}")

print("\n" + "=" * 78)
print("CONCLUSION")
print("=" * 78)
for row_name, (max_diff, model_name, bench_name, macro) in winners.items():
    verdict = "EXACT" if max_diff < TOLERANCE else f"closest (maxdiff={max_diff:.4f})"
    print(f"  {row_name} -> {model_name}  on {bench_name}  [{verdict}]")

distinct = {w[1] for w in winners.values()}
if len(distinct) == 1:
    print(f"\n  All reported rows come from a single model: {distinct.pop()}")
else:
    print(f"\n  WARNING: rows map to multiple models: {distinct}")
